# SDN Binary Attack Classifier — Multi-Model LLM Evaluation

Tests the final SDN security prompt (simplified to binary output) across multiple cloud LLM models.

- **label = 1** → attack detected (ip / rrm / both)
- **label = 0** → no attack (do_nothing)

Metrics computed: **Accuracy, Precision, Recall, F1**  
Output saved to: `test_result_paper.csv`

## Shared Setup — Imports, LLM Functions, Prompt, Test Dataset

In [1]:
import json
import csv
import time
import requests
from datetime import datetime

try:
    from sklearn.metrics import (
        precision_score, recall_score, f1_score,
        accuracy_score, confusion_matrix,
    )
    SKLEARN_AVAILABLE = True
except ImportError:
    SKLEARN_AVAILABLE = False
    print('[WARN] sklearn not found — using manual metric computation')

CLOUD_URL = 'https://ollama.com/api/chat'
API_KEY   = '23fbf0f676584a7983158ded37540f2c.9C4Aw8pI8jXQlC1vDCGIF7nb'

# ── Reused from module_llm_helper.py ────────────────────────────────────────

def call_cloud_llm(prompt, model_name):
    payload = {
        'model': model_name,
        'messages': [
            {'role': 'system', 'content': 'Return ONLY valid JSON. No text.'},
            {'role': 'user',   'content': prompt},
        ],
        'stream': False,
        'options': {'temperature': 0, 'top_p': 0.9},
    }
    headers = {
        'Authorization': f'Bearer {API_KEY}',
        'Content-Type':  'application/json',
    }
    start = time.time()
    r = requests.post(CLOUD_URL, headers=headers, json=payload, timeout=300)
    r.raise_for_status()
    latency = time.time() - start
    out = r.json().get('message', {}).get('content', '').strip()
    return out, latency

def get_label(prompt, model_name, max_retries=3):
    """Call LLM and extract label=0|1. Reprompts if label is missing or invalid."""
    last_out, last_latency = '', 0.0
    for attempt in range(1, max_retries + 1):
        try:
            out, latency = call_cloud_llm(prompt, model_name)
            last_out, last_latency = out, latency
            parsed = json.loads(out)
            label  = int(parsed.get('label', -1))
            if label in (0, 1):
                return label, out, latency
            print(f'  [RETRY {attempt}/{max_retries}] invalid label={label}, reprompting...')
        except json.JSONDecodeError:
            print(f'  [RETRY {attempt}/{max_retries}] invalid JSON, reprompting...')
        except Exception as exc:
            print(f'  [RETRY {attempt}/{max_retries}] error: {exc}, reprompting...')
    return -1, last_out, last_latency

# ── Prompt — original logic, binary label output ─────────────────────────────

def build_binary_prompt(scene: dict) -> str:
    return f"""
You are the final SDN security decision layer.

You receive reports from previous LLM layers about suspected hosts and suspected links.
Your task is to decide the final Moving Target Defense action.

Available decisions:
- "ip": use IP mutation for host flooding / host-side DoS
- "rrm": use route mutation for link flooding / link exhaustion
- "both": use both IP mutation and route mutation
- "do_nothing": no action

Definitions:
- Host flooding means a host is receiving or sending abnormally high packets per second.
- Use host rx_pps and tx_pps to detect host flooding.
- Link flooding means a link is near or above the bandwidth limit.
- The link bandwidth limit is 9 Mbps = 9000 kbps.
- Use link rx_kbps and tx_kbps to detect link congestion.

Decision rules:
1. If only host flooding is present, choose "ip".
2. If only link flooding is present, choose "rrm".
3. If both host flooding and link flooding are present, choose "both".
4. If neither host flooding nor link flooding is clearly present, choose "do_nothing".

RRM rule:
RRM is used when an attacker congests a link near or above 9000 kbps and risks breaking the connection.
For RRM, select the congested link IDs and the active/causing host MACs related to those links when available.

IP rule:
IP mutation is used when a host is directly flooded with high pps traffic.
Select the targeted or flooded host MACs.

Do-not-overreact rule:
Do not choose "ip" only because a host is listed as suspected.
Do not choose "rrm" only because a link is listed as suspected.
Choose an action only when telemetry supports it.
If links are below capacity and host pps is not abnormal, choose "do_nothing".

Input:
{json.dumps(scene, ensure_ascii=False)}

Based on the input and the rules above, output a binary label:
- label=1 if you would choose "ip", "rrm", or "both" (an attack is present)
- label=0 if you would choose "do_nothing" (no attack)

Return strict JSON only. No markdown. No extra text. No comments. No trailing commas.

{{"label": 1}}
or
{{"label": 0}}
""".strip()

# ── Dataset loading from tet_dataset.csv ─────────────────────────────────────

def build_scene_from_candidates(host_candidates, link_candidates):
    """Convert candidate trend arrays into a scene dict using peak (max) values."""
    suspected_hosts = []
    for h in host_candidates:
        suspected_hosts.append({
            'mac':     h['candidate_id'],
            'rx_pps':  max(h['rx_pps_trend']),
            'tx_pps':  max(h['tx_pps_trend']),
            'rx_kbps': max(h['rx_kbps_trend']),
            'tx_kbps': max(h['tx_kbps_trend']),
        })
    suspected_links = []
    for l in link_candidates:
        suspected_links.append({
            'link_id': l['candidate_id'],
            'rx_pps':  max(l['rx_pps_trend']),
            'tx_pps':  max(l['tx_pps_trend']),
            'rx_kbps': max(l['rx_kbps_trend']),
            'tx_kbps': max(l['tx_kbps_trend']),
        })
    return {'suspected_hosts': suspected_hosts, 'suspected_links': suspected_links}

def load_test_cases(csv_path='tet_dataset.csv', max_samples=None):
    """Load from tet_dataset.csv — columns: scene (JSON), assessment, label (0/1)."""
    cases = []
    with open(csv_path, newline='', encoding='utf-8') as f:
        for i, row in enumerate(csv.DictReader(f)):
            raw_scene = json.loads(row['scene'])
            scene = build_scene_from_candidates(
                raw_scene.get('host_candidates', []),
                raw_scene.get('link_candidates', []),
            )
            cases.append({
                'scene_id':     raw_scene.get('scene_id', i + 1),
                'ground_truth': int(row['label']),
                'assessment':   row['assessment'],
                'scene':        scene,
            })
            if max_samples and len(cases) >= max_samples:
                break

    from collections import Counter
    dist = Counter(c['ground_truth'] for c in cases)
    print(f'Loaded {len(cases)} test cases from {csv_path}')
    print(f'  Attack  (1): {dist[1]}')
    print(f'  No-attack (0): {dist[0]}')
    return cases

# Load dataset — set max_samples=N to limit for a quick run (e.g. max_samples=50)
TEST_CASES = load_test_cases('tet_dataset.csv', max_samples=None)

# ── Metrics helper ───────────────────────────────────────────────────────────

def compute_metrics(y_true, y_pred):
    if SKLEARN_AVAILABLE:
        cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
        tn, fp, fn, tp = cm.ravel() if cm.shape == (2, 2) else (0, 0, 0, 0)
        return {
            'accuracy':  round(float(accuracy_score(y_true, y_pred)),                   4),
            'precision': round(float(precision_score(y_true, y_pred, zero_division=0)), 4),
            'recall':    round(float(recall_score(y_true, y_pred, zero_division=0)),    4),
            'f1':        round(float(f1_score(y_true, y_pred, zero_division=0)),        4),
            'tp': int(tp), 'fp': int(fp), 'fn': int(fn), 'tn': int(tn),
        }
    tp = sum(1 for t, p in zip(y_true, y_pred) if t == 1 and p == 1)
    fp = sum(1 for t, p in zip(y_true, y_pred) if t == 0 and p == 1)
    fn = sum(1 for t, p in zip(y_true, y_pred) if t == 1 and p == 0)
    tn = sum(1 for t, p in zip(y_true, y_pred) if t == 0 and p == 0)
    prec = tp / (tp + fp)        if (tp + fp) > 0 else 0.0
    rec  = tp / (tp + fn)        if (tp + fn) > 0 else 0.0
    f1   = 2*prec*rec/(prec+rec) if (prec + rec) > 0 else 0.0
    acc  = (tp + tn) / len(y_true) if y_true else 0.0
    return {'accuracy': round(acc,4), 'precision': round(prec,4),
            'recall': round(rec,4), 'f1': round(f1,4),
            'tp': tp, 'fp': fp, 'fn': fn, 'tn': tn}

# ── Runner for one model ─────────────────────────────────────────────────────

def run_model(model_name: str) -> tuple:
    """Run all TEST_CASES against model_name. Returns (rows, metrics)."""
    rows, y_true, y_pred = [], [], []
    run_ts = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    n = len(TEST_CASES)

    print(f'\n{"-"*60}')
    print(f'  Model : {model_name}')
    print(f'  Start : {run_ts}')
    print(f'  Cases : {n}')
    print(f'{"-"*60}')

    for i, case in enumerate(TEST_CASES, 1):
        scene_id     = case['scene_id']
        ground_truth = case['ground_truth']
        scene        = case['scene']

        print(f'  [{i:04d}/{n}] scene={scene_id:<4} GT={ground_truth}', end='  ')
        prompt    = build_binary_prompt(scene)
        raw_out   = ''
        predicted = -1
        latency   = 0.0
        error_msg = ''

        try:
            predicted, raw_out, latency = get_label(prompt, model_name)
        except Exception as exc:
            error_msg = str(exc)
            predicted = -1

        correct = (predicted == ground_truth) if predicted != -1 else False
        status  = '✓' if correct else ('ERR' if predicted == -1 else '✗')
        print(f'Pred={predicted}  {latency:.2f}s  [{status}]')

        rows.append({
            'model':         model_name,
            'scene_id':      scene_id,
            'ground_truth':  ground_truth,
            'predicted':     predicted,
            'correct':       int(correct) if predicted != -1 else 'N/A',
            'latency_s':     round(latency, 3),
            'run_timestamp': run_ts,
            'raw_response':  raw_out,
            'error':         error_msg,
        })
        if predicted != -1:
            y_true.append(ground_truth)
            y_pred.append(predicted)

    metrics = compute_metrics(y_true, y_pred) if y_true else {}
    print(f'\n  Valid: {len(y_true)}/{n} | '
          f'Acc={metrics.get("accuracy","?")}  '
          f'P={metrics.get("precision","?")}  '
          f'R={metrics.get("recall","?")}  '
          f'F1={metrics.get("f1","?")}')
    return rows, metrics

# Storage for all model results (populated by subsequent cells)
ALL_RESULTS = {}   # model_name -> (rows, metrics)

print('\nSetup complete. Run each model cell below.')


Loaded 863 test cases from tet_dataset.csv
  Attack  (1): 461
  No-attack (0): 402

Setup complete. Run each model cell below.


In [2]:
# ── Quick 10-sample sanity check ─────────────────────────────────────────────
# Run this before the full model cells to verify the API, prompt, and CSV all work.

SAMPLE_10 = TEST_CASES[:10]
MODEL_TEST = 'gpt-oss:20b-cloud'

print(f'Running 10-sample test with {MODEL_TEST}...\n')
rows_test, metrics_test = [], []
y_t, y_p = [], []

for i, case in enumerate(SAMPLE_10, 1):
    print(f'  [{i:02d}/10] scene={case["scene_id"]}  GT={case["ground_truth"]}', end='  ')
    prompt    = build_binary_prompt(case['scene'])
    predicted, raw_out, latency = get_label(prompt, MODEL_TEST)
    correct   = (predicted == case['ground_truth']) if predicted != -1 else False
    status    = '✓' if correct else ('ERR' if predicted == -1 else '✗')
    print(f'Pred={predicted}  {latency:.2f}s  [{status}]')
    rows_test.append({
        'model': MODEL_TEST, 'scene_id': case['scene_id'],
        'ground_truth': case['ground_truth'], 'predicted': predicted,
        'correct': int(correct) if predicted != -1 else 'N/A',
        'latency_s': round(latency, 3), 'raw_response': raw_out, 'error': '',
    })
    if predicted != -1:
        y_t.append(case['ground_truth'])
        y_p.append(predicted)

if y_t:
    m = compute_metrics(y_t, y_p)
    print(f'\n  Acc={m["accuracy"]}  P={m["precision"]}  R={m["recall"]}  F1={m["f1"]}')
    print(f'  TP={m["tp"]} FP={m["fp"]} FN={m["fn"]} TN={m["tn"]}')


Running 10-sample test with gpt-oss:20b-cloud...

  [01/10] scene=1  GT=1  Pred=1  1.66s  [✓]
  [02/10] scene=2  GT=1  Pred=1  1.15s  [✓]
  [03/10] scene=3  GT=1  Pred=1  3.51s  [✓]
  [04/10] scene=4  GT=0  Pred=0  0.92s  [✓]
  [05/10] scene=5  GT=1  Pred=1  2.28s  [✓]
  [06/10] scene=6  GT=1  Pred=1  2.33s  [✓]
  [07/10] scene=7  GT=0  Pred=0  1.24s  [✓]
  [08/10] scene=8  GT=0  Pred=0  1.29s  [✓]
  [09/10] scene=9  GT=1  Pred=1  2.21s  [✓]
  [10/10] scene=10  GT=1  Pred=1  2.01s  [✓]

  Acc=1.0  P=1.0  R=1.0  F1=1.0
  TP=7 FP=0 FN=0 TN=3


---
## Model 1 — `gpt-oss:20b-cloud`

In [3]:
MODEL_1 = 'gpt-oss:20b-cloud'

rows_1, metrics_1 = run_model(MODEL_1)
ALL_RESULTS[MODEL_1] = (rows_1, metrics_1)


------------------------------------------------------------
  Model : gpt-oss:20b-cloud
  Start : 2026-05-03 19:43:01
  Cases : 863
------------------------------------------------------------
  [0001/863] scene=1    GT=1  Pred=1  2.17s  [✓]
  [0002/863] scene=2    GT=1  Pred=1  2.38s  [✓]
  [0003/863] scene=3    GT=1  Pred=1  3.07s  [✓]
  [0004/863] scene=4    GT=0  Pred=0  1.27s  [✓]
  [0005/863] scene=5    GT=1  Pred=1  1.47s  [✓]
  [0006/863] scene=6    GT=1  Pred=1  2.03s  [✓]
  [0007/863] scene=7    GT=0  Pred=0  1.40s  [✓]
  [0008/863] scene=8    GT=0  Pred=0  0.75s  [✓]
  [0009/863] scene=9    GT=1  Pred=1  1.51s  [✓]
  [0010/863] scene=10   GT=1  Pred=1  3.91s  [✓]
  [0011/863] scene=11   GT=1  Pred=1  2.09s  [✓]
  [0012/863] scene=12   GT=0  Pred=0  1.17s  [✓]
  [0013/863] scene=13   GT=1  Pred=1  1.51s  [✓]
  [0014/863] scene=14   GT=0  Pred=0  1.01s  [✓]
  [0015/863] scene=15   GT=0  Pred=0  1.03s  [✓]
  [0016/863] scene=16   GT=0  Pred=0  1.60s  [✓]
  [0017/863] scene=17

---
## Model 2 — `qwen3.5:cloud`

In [4]:
MODEL_2 = 'qwen3.5:cloud'

rows_2, metrics_2 = run_model(MODEL_2)
ALL_RESULTS[MODEL_2] = (rows_2, metrics_2)


------------------------------------------------------------
  Model : qwen3.5:cloud
  Start : 2026-05-03 20:20:05
  Cases : 863
------------------------------------------------------------
  [0001/863] scene=1    GT=1    [RETRY 1/3] error: 403 Client Error: Forbidden for url: https://ollama.com/api/chat, reprompting...
  [RETRY 2/3] error: 403 Client Error: Forbidden for url: https://ollama.com/api/chat, reprompting...
  [RETRY 3/3] error: 403 Client Error: Forbidden for url: https://ollama.com/api/chat, reprompting...
Pred=-1  0.00s  [ERR]
  [0002/863] scene=2    GT=1    [RETRY 1/3] error: 403 Client Error: Forbidden for url: https://ollama.com/api/chat, reprompting...
  [RETRY 2/3] error: 403 Client Error: Forbidden for url: https://ollama.com/api/chat, reprompting...
  [RETRY 3/3] error: 403 Client Error: Forbidden for url: https://ollama.com/api/chat, reprompting...
Pred=-1  0.00s  [ERR]
  [0003/863] scene=3    GT=1    [RETRY 1/3] error: 403 Client Error: Forbidden for url: https

KeyboardInterrupt: 

---
## Model 3 — `gemma4:26b-cloud`

In [5]:
MODEL_3 = 'gemma4:26b-cloud'

rows_3, metrics_3 = run_model(MODEL_3)
ALL_RESULTS[MODEL_3] = (rows_3, metrics_3)


------------------------------------------------------------
  Model : gemma4:26b-cloud
  Start : 2026-05-03 20:21:40
  Cases : 863
------------------------------------------------------------
  [0001/863] scene=1    GT=1    [RETRY 1/3] error: 404 Client Error: Not Found for url: https://ollama.com/api/chat, reprompting...
  [RETRY 2/3] error: 404 Client Error: Not Found for url: https://ollama.com/api/chat, reprompting...
  [RETRY 3/3] error: 404 Client Error: Not Found for url: https://ollama.com/api/chat, reprompting...
Pred=-1  0.00s  [ERR]
  [0002/863] scene=2    GT=1    [RETRY 1/3] error: 404 Client Error: Not Found for url: https://ollama.com/api/chat, reprompting...
  [RETRY 2/3] error: 404 Client Error: Not Found for url: https://ollama.com/api/chat, reprompting...
  [RETRY 3/3] error: 404 Client Error: Not Found for url: https://ollama.com/api/chat, reprompting...
Pred=-1  0.00s  [ERR]
  [0003/863] scene=3    GT=1    [RETRY 1/3] error: 404 Client Error: Not Found for url: ht

KeyboardInterrupt: 

---
## Model 4 — `Gemma 31b:cloud`

In [10]:
# MODEL_4 = 'deepseek-v3.2:cloud'
MODEL_4 = 'gemma4:31b-cloud'

rows_4, metrics_4 = run_model(MODEL_4)
ALL_RESULTS[MODEL_4] = (rows_4, metrics_4)


------------------------------------------------------------
  Model : gemma4:31b-cloud
  Start : 2026-05-03 21:42:41
  Cases : 863
------------------------------------------------------------
  [0001/863] scene=1    GT=1  

KeyboardInterrupt: 

---
## Results Summary & Export to `test_result_paper.csv`

In [8]:
OUTPUT_CSV = 'test_result_paper.csv'

sample_fields = [
    'model', 'test_id', 'scenario_type', 'scenario_category',
    'ground_truth', 'predicted', 'correct',
    'latency_s', 'run_timestamp', 'raw_response', 'error',
]

with open(OUTPUT_CSV, 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=sample_fields, extrasaction='ignore')
    writer.writeheader()

    for model_name, (rows, _) in ALL_RESULTS.items():
        writer.writerows(rows)
        writer.writerow({k: '' for k in sample_fields})  # blank separator

    # ── Metrics summary block ─────────────────────────────────────────────
    writer.writerow({k: '' for k in sample_fields})
    writer.writerow({
        'model': '=== METRICS SUMMARY ===',
        **{k: '' for k in sample_fields if k != 'model'}
    })
    summary_header = {
        'model':            'model',
        'test_id':          'accuracy',
        'scenario_type':    'precision',
        'scenario_category':'recall',
        'ground_truth':     'f1',
        'predicted':        'tp',
        'correct':          'fp',
        'latency_s':        'fn',
        'run_timestamp':    'tn',
        'raw_response':     '',
        'error':            '',
    }
    writer.writerow(summary_header)
    for model_name, (_, metrics) in ALL_RESULTS.items():
        writer.writerow({
            'model':            model_name,
            'test_id':          metrics.get('accuracy',  ''),
            'scenario_type':    metrics.get('precision', ''),
            'scenario_category':metrics.get('recall',    ''),
            'ground_truth':     metrics.get('f1',        ''),
            'predicted':        metrics.get('tp',        ''),
            'correct':          metrics.get('fp',        ''),
            'latency_s':        metrics.get('fn',        ''),
            'run_timestamp':    metrics.get('tn',        ''),
            'raw_response':     '',
            'error':            '',
        })

print(f'Saved → {OUTPUT_CSV}\n')

# ── Print comparison table ────────────────────────────────────────────────
print(f'{"Model":<30} {"Acc":>6} {"Prec":>6} {"Rec":>6} {"F1":>6}  TP FP FN TN')
print('-' * 72)
for model_name, (_, m) in ALL_RESULTS.items():
    print(
        f'{model_name:<30} '
        f'{m.get("accuracy","-"):>6} '
        f'{m.get("precision","-"):>6} '
        f'{m.get("recall","-"):>6} '
        f'{m.get("f1","-"):>6}  '
        f'{m.get("tp","-"):>2} '
        f'{m.get("fp","-"):>2} '
        f'{m.get("fn","-"):>2} '
        f'{m.get("tn","-"):>2}'
    )

Saved → test_result_paper.csv

Model                             Acc   Prec    Rec     F1  TP FP FN TN
------------------------------------------------------------------------
gpt-oss:20b-cloud              0.8957 0.8616 0.9588 0.9076  442 71 19 331
gemma4:31b-cloud               0.8017 0.8778 0.7231  0.793  316 44 121 351
